# le bestiaire spectral 
$\rightarrow$ **mesurer la température par ajustement de la courbe de Planck et en déduire la gravité de surface (log(g))**


# le cours
## le diagramme HR et la distribution spectrale
- Les étoiles se présentent dans différentes couleurs déterminées par leur température.
- Les étoiles chaudes sont bleues, tandis que celles plus froides sont rouges.
- Dans un ordre croissant de température, une étoile est rouge, orange, jaune, blanche, bleue ou violette.
- Plus un corps est chaud, plus les photons qui s'en échappent ont d'énergie, et plus leur longueur d'onde est faible.

<!--  <img src="hr_diag_01.jpeg" alt="diagramme HR" width="1000" height="800">  -->
<img src="images/hr_diag_02.gif" alt="diagramme HR" width="490" height="390"> <img src="Black_body.png" alt="rayonnement du corps noir" width="445" height="200">

(source : https://media4.obspm.fr et wikipedia)

Pour un même type spectral, la classe de luminosité (de I à V) détermine leur intensité lumineuse :
- Naines (V) : Petites, gravité de surface importante $\rightarrow$ Pression élevée $\rightarrow$ ; Les atomes se cognent souvent $\rightarrow$ Raies larges (élargissement collisionnel).
- Supergéantes (I) : Immenses, atmosphère très diluée $\rightarrow$ Pression faible $\rightarrow$ Raies fines et profondes.

La formule de Planck décrit la distribution spectrale de l'énergie émise par un corps noir en fonction de la fréquence ou de la longueur d'onde, à une température donnée : 
  
$$B_\lambda(T) = \underbrace{\frac{2hc^2}{\lambda^5}}_{\text{Terme 1}} \cdot \underbrace{\frac{1}{e^{\frac{hc}{\lambda k_B T}} - 1}}_{\text{Terme 2}}$$

- Aux courtes longueurs d'onde (Bleu / UV) : Le terme $\frac{1}{\lambda^5}$ devient gigantesque.
- Heureusement, le Terme 2 (l'exponentielle) intervient : $\exp(1/\lambda)$ devient encore plus grand que $\lambda^5$.
- Comme il est au dénominateur, il "écrase" tout le résultat vers zéro. C'est ce qui empêche la "catastrophe ultraviolette".
- Aux grandes longueurs d'onde (Rouge / Infra-rouge) : L'exponentielle devient négligeable (proche de 1).
- La Température : plus $T$ est élevée, plus le pic de la courbe se déplace vers le bleu (Loi de Wien) et plus l'intensité totale augmente (Loi de Stefan-Boltzmann).

**ATTENTION** : pour les étoiles chaudes (O, B, A), le spectre observé est souvent plus "rouge" (pente moins raide) que la théorie, à cause de la poussière située entre l'étoile et nous. Pour corriger cela, on introduit l'Excès de couleur, noté $E(B-V)$ :
- $E(B-V) = 0$ : Pas de poussière (cas idéal).
- $E(B-V) > 0$ : Il y a de la poussière qui absorbe le bleu.

## la gravité
On obtient la gravité de surface avec : $g = G \frac{M}{R^2}$ 

On utilise plutôt le log(g) pour prendre en compte les grandes différences de valeurs : 
- Supergéante (Classe I) $\rightarrow$ Très grand rayon, faible densité.$0.0 \text{ à } 2.0$
- Géante (Classe III) $\rightarrow$ Rayon intermédiaire, basse densité.$2.5 \text{ à } 3.5$
- Naine (Séquence Principale, Classe V) $\rightarrow$ Étoiles compactes (comme le Soleil).$4.0 \text{ à } 5.0$
- Naine Blanche $\rightarrow$ Très petit rayon, densité extrême.$7.0 \text{ à } 9.0$




# les mesures

# lancement du dashboard spectro 

- lancer la cellule suivante
- souris sur '**Colormap**', bouton droit, menu "**create new view for cell output**"
- déplacer le nouvel onglet créé '**Output View**' pour le garder visible pendant que vous naviguez et exécutez les cellules de code

En cas de souci d'affichage $\rightarrow$  '**CTRL-R**'


In [1]:
%matplotlib widget
from spectra_widget import SpectraWidget

db = SpectraWidget()
db.show()


## les étoiles

### Type O: Chaudes et Bleues
- Température : $> 30\,000 \text{ K}$
- Le continuum bleu est très intense.
- L'hydrogène est faible car il est presque totalement ionisé.
- Le Marqueur : Présence d'Hélium ionisé (He II 4200, 4542, 4686), signature exclusive des étoiles O.
- exemples :
    - Naine (V)	10 Lacertae (O9V)	Raies d'absorption de l'Hélium visibles mais "floues" à cause de la pression.



In [3]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()
#filename = 'data/MilesLibraryBASS/09Ib s0252 HD057061.dat'
#_spc_array = np.loadtxt(filename)
#_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))

filename = 'data/plouis/c_10Lac.fit'
#filename = 'data/plouis/c_Vega.fit'

_spec1d = Spectrum.read(filename)
_spec1d = _spec1d [3800*u.AA : 8000*u.AA]
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='10Lac')



INFO: affichage du spectre '10Lac' : 4200 pts, X:[3800.0:7999.0]


In [10]:
# calcul de la température
import numpy as np
import matplotlib.pyplot as plt
from astropy import constants as const
from specutils import Spectrum
from scipy.optimize import curve_fit

def planck_temperature(spectrum, lam_min = 4000.0 , lam_max = 6500.0 ):
    _lam = spectrum.wavelength.value
    _flux = spectrum.flux.value

    # on ne garde que le range demandé
    mask = (_lam >= lam_min) & (_lam <= lam_max)
    lam = _lam[mask]
    flux = _flux[mask]
    
    # la loi de Planck
    def planck_lambda(lam_A, T):
        lam = lam_A * 1e-10  # Å -> m
        h = const.h.value
        c = const.c.value
        k = const.k_B.value
        x = h*c/(lam*k*T)
        return (2*h*c**2 / lam**5) / (np.exp(x) - 1)
    
    # la mise à l'echelle 
    def fit_func(lam_A, T, scale):
        return scale * planck_lambda(lam_A, T)
    
    # la Normalisation
    flux_n = flux / np.nanmax(flux)    
    
    # l'ajustement
    popt, pcov = curve_fit(fit_func, lam, flux_n, p0=[8000, 1e-20])
    best_T, best_scale = popt

    perr = np.sqrt(np.diag(pcov))
    incertitude_T = perr[0] # Erreur sur la Température

    print(f"Température ajustée : {best_T:.0f} K +/-{incertitude_T:.0f}")
    
    return (best_T, lam, flux_n, fit_func(lam, best_T, best_scale))
    
    

In [11]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=3900)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')


Température ajustée : 26229 K +/-217
INFO: affichage du spectre 'Température ajustée : 26229 K' : 2601 pts, X:[3900.0:6500.0]
INFO: affichage du spectre 'étoile' : 4200 pts, X:[3800.0:7999.0]



### Type B : L'Hélium Neutre
- Température : $10\,000 - 30\,000 \text{ K}$
- Les raies de l'Hydrogène (Balmer) deviennent plus visibles.
- Le Marqueur : Maximum de l'Hélium neutre ($\text{He I}$) (4026, 4471, 5876, 6678) : Signature des étoiles B.
- exemples:
    - Naine (V)Regulus (B7V) Raies d'Hélium larges.
    - Supergéante (I)Rigel (B8I) Raies d'Hélium beaucoup plus fines et nettes. C'est l'exemple classique pour montrer l'effet de pression.

In [12]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()
_spc_array = np.loadtxt('data/MilesLibraryBASS/B9.5V s0823 HD209459.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='B9.5V')

_spc_array = np.loadtxt('data/MilesLibraryBASS/B9 s0430 HD105262.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='B9')


INFO: affichage du spectre 'B9.5V' : 4367 pts, X:[3500.0:7429.4]
INFO: affichage du spectre 'B9' : 4367 pts, X:[3500.0:7429.4]


In [16]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=3900)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')


Température ajustée : 12159 K +/-104
INFO: affichage du spectre 'Température ajustée : 12159 K' : 2889 pts, X:[3900.5:6499.7]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Type A : Le Royaume de l'Hydrogène
- Température : $7\,500 - 10\,000 \text{ K}$
- Le Marqueur : Les raies de l'Hydrogène (Série de Balmer : $H\alpha, H\beta, H\gamma$) sont à leur intensité maximale.
- Exemples:
    - Naine (V)	Véga ou Sirius (A0V/A1V)	Les raies de l'hydrogène sont très larges avec des "ailes" étendues (broad wings).
    - Supergéante (I)	Deneb (A2I)	Les raies de l'hydrogène sont étroites et profondes, sans les ailes larges typiques des naines.

In [17]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spc_array = np.loadtxt('data/MilesLibraryBASS/A0 s0085 HD014829.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='AO')

_spc_array = np.loadtxt('data/MilesLibraryBASS/A0 V (HB) s0446 HD109995.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='A0V')


INFO: affichage du spectre 'AO' : 4367 pts, X:[3500.0:7429.4]
INFO: affichage du spectre 'A0V' : 4367 pts, X:[3500.0:7429.4]


In [18]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=4100)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')


Température ajustée : 12705 K +/-114
INFO: affichage du spectre 'Température ajustée : 12705 K' : 2667 pts, X:[4100.3:6499.7]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Type F : La Transition Métallique
- Température : $6\,000 - 7\,500 \text{ K}$
- Le Marqueur : L'hydrogène faiblit.
- Le Calcium ionisé ($\text{Ca II}$, raies H et K) devient fort.
- Les métaux neutres ($\text{Fe I}, \text{Cr}$) apparaissent.
- exemples :
    - Naine (V)	Procyon (F5V)	Spectre "propre" avec des métaux modérés.
    - Supergéante (I)	Canopus (F0I)	Les raies de métaux ionisés (Ti II, Fe II) sont renforcées par rapport aux naines.


In [19]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spc_array = np.loadtxt('data/MilesLibraryBASS/F0 s0345 HD081029.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='F0')

_spc_array = np.loadtxt('data/MilesLibraryBASS/F0 V s0910 HD029375.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='F0V')


INFO: affichage du spectre 'F0' : 4367 pts, X:[3500.0:7429.4]
INFO: affichage du spectre 'F0V' : 4367 pts, X:[3500.0:7429.4]


In [20]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=4350)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')


Température ajustée : 8786 K +/-30
INFO: affichage du spectre 'Température ajustée : 8786 K' : 2389 pts, X:[4350.5:6499.7]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Type G : Les Étoiles de Type Solaire
- Température : $5\,000 - 6\,000 \text{ K}$
- Le Marqueur : Les raies H et K du $\text{Ca II}$ sont énormes.
- La Bande G (molécule CH) est très visible.
- exemples :
    - Naine (V)	Soleil (G2V)	La bande moléculaire CN (Cyanogène) est faible.
    - Géante (III)	Capella (G5III)	La bande du Cyanogène (CN) est forte dans le violet. C'est un critère discriminant majeur pour les géantes G et K.

In [21]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spc_array = np.loadtxt('data/MilesLibraryBASS/G0V s0176 HD034411.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('mJy'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='G0V')


INFO: affichage du spectre 'G0V' : 4367 pts, X:[3500.0:7429.4]


In [30]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=5500)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')


Température ajustée : 6384 K +/-34
INFO: affichage du spectre 'Température ajustée : 6384 K' : 1111 pts, X:[5500.7:6499.7]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Type K : L'Arrivée des Molécules
- Température : $3\,500 - 5\,000 \text{ K}$
- Le Marqueur : Forêt de raies métalliques. Apparition des bandes moléculaires d'oxyde de titane ($\text{TiO}$), TiO (4765, 6158, 7050) et Mg I (5175)
- exemples :
    - Naine (V)	61 Cygni (K5V)	Raies métalliques très encombrées.
    - Géante (III)	Arcturus (K1III)	Comme pour les G, le CN est plus fort. Le profil global semble plus "accidenté" dans le bleu.

In [31]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spc_array = np.loadtxt('data/MilesLibraryBASS/K0 s0187 HD037828.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='K0')

#_spc_array = np.loadtxt('data/MilesLibraryBASS/K0 V s0724 HD185144.dat')
#_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
#db.load_spectrum(_spec1d.wavelength, _spec1d.flux, label='K0V')


INFO: affichage du spectre 'K0' : 4367 pts, X:[3500.0:7429.4]


In [34]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=4400, lam_max=7400)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')



Température ajustée : 4506 K +/-4
INFO: affichage du spectre 'Température ajustée : 4506 K' : 3334 pts, X:[4400.0:7399.7]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Type M : Les Étoiles Froides
- Température : $< 3\,500 \text{ K}$
- Le Marqueur : Le spectre est "mangé" par d'énormes bandes d'absorption moléculaires ($\text{TiO}$), donnant un aspect cannelé (en dents de scie).
- exemples :
    - Naine (V) Proxima Centauri (M5V) Présence de bandes de CaH (Hydrure de Calcium) et parfois $\text{MgH}$. C'est la signature de la haute gravité des naines rouges.
    - Supergéante (I) Bételgeuse (M1-M2I) Bandes de TiO énormes, mais absence quasi-totale de CaH.

In [35]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spc_array = np.loadtxt('data/MilesLibraryBASS/M0V s0339 HD079211.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='M0V')

_spc_array = np.loadtxt('data/MilesLibraryBASS/M0.5 IIb s0535 HD132933.dat')
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='M05II')


INFO: affichage du spectre 'M0V' : 4367 pts, X:[3500.0:7429.4]
INFO: affichage du spectre 'M05II' : 4367 pts, X:[3500.0:7429.4]


In [37]:
# récupère la température

best_T, lam, flux_n, fit = planck_temperature(_spec1d,lam_min=4000, lam_max=7000)

db.clear_spectra()
db.show_spectrum(lam, fit, label=f"Température ajustée : {best_T:.0f} K", ls='--', color='red')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux / np.nanmax(_spec1d.flux), label='étoile')



Température ajustée : 3530 K +/-7
INFO: affichage du spectre 'Température ajustée : 3530 K' : 3333 pts, X:[4000.4:6999.2]
INFO: affichage du spectre 'étoile' : 4367 pts, X:[3500.0:7429.4]


### Méthode d'identification rapide
1. Regarder l'Hydrogène : Fort $\rightarrow$ A.
2. Regarder l'Hélium : Présent $\rightarrow$ O ou B.
3. Regarder les bandes moléculaires : Présentes $\rightarrow$ K ou M.
4. Pour la luminosité (I vs V) : Regarder la largeur des raies (surtout pour B et A) ou la présence d'éléments sensibles à la gravité comme le Sr II ou le CN (pour F, G, K).


ATTENTION au "Piège" des Naines Rouges (Type M):
- géantes rouges (comme Bételgeuse) et petites naines rouges (comme l'étoile de Barnard) ont toutes deux des bandes de TiO.
- regarder dans le rouge profond (~6380-6800 $\mathring{A}$) : Si bandes de CaH (Hydrure de Calcium) -> c'est une petite étoile compacte (Naine) car le CaH survit mal dans la faible gravité des géantes.


Sr II (4078) : La raie du Strontium :
- Forte = Supergéante (Basse pression).
- Faible = Naine (Haute pression).

CaH (6385) : Hydrure de Calcium :
- Présent = Naine Rouge (M V).
- Absent = Géante Rouge (M III).



## gravité (log(g)


<img src="images/logg_vs_temp.jpg" alt="diagramme HR" width="1600" height="1500">

Formule empirique :$$\log(g) \approx 4.1 + \frac{EW - 16}{4.5} - 0.15 \times \frac{T - 9600}{1000}$$


## les nebuleuses

In [38]:
from astropy import units as u
from specutils import Spectrum

db.clear_spectra()

_spec1d = Spectrum.read('data/plouis/_m57_20240710_865.fit')
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='M57')

#_spec1d = Spectrum.read('data/plouis/ngc6543_20161006_849.fit')
#db.load_spectrum(_spec1d.wavelength, _spec1d.flux, label='ngc6543')


INFO: affichage du spectre 'M57' : 3235 pts, X:[4349.9:7187.7]


## les galaxies

In [39]:
db.clear_spectra()

_spc_array = np.loadtxt('data/plouis/_ngc4151_20170121_974_P.Louis.dat')

_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='ngc4151')

#_spc_array = np.loadtxt('data/plouis/_ngc7215-_20161228_896_P.Louis.dat')
#_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
#db.load_spectrum(_spec1d.wavelength, _spec1d.flux, label='ngc7215')

#_spc_array = np.loadtxt('data/plouis/_3c273_20170407_870_PLouis.dat')
#_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('adu'))
#db.load_spectrum(_spec1d.wavelength, _spec1d.flux, label='3c273')

# on affiche des barres verticales pour marquer les raies de H
#h_a = db.ax_spec.axvline(x=6563, color='red', linestyle='--', alpha=0.6, linewidth=4)
#h_b = db.ax_spec.axvline(x=4861, color='red', linestyle='--', alpha=0.6, linewidth=4)

#h_a.remove()
#h_b.remove()


INFO: affichage du spectre 'ngc4151' : 3499 pts, X:[4244.5:7071.7]
